# 05 協調支援の最適化

予算 K 本の国家ペアにだけ、協調支援（対話・協定・信頼形成・経済協力のどれか）を行えるとき、どのペアを選ぶか（RQ6、Comparison B）。

| | 全探索 | Greedy 法 |
| --- | --- | --- |
| 方法 | $C(E, K)$ 通りの組合せをすべて評価 | 1 本ずつ、改善が最大のエッジを足す |
| 評価回数 | $C(E, K)$ | $E + (E-1) + \dots + (E-K+1)$ |
| 最適性 | この候補集合・評価方法の中で保証 | 保証なし |

どちらも**同じ候補集合・同じ予算・同じ評価関数**（30 回 × 30 年のモンテカルロ、共通乱数）を使います。**仮想的な政策操作**であり、現実の外交政策の有効性を表すものではありません。

> **補遺トピック G2・教育用の抽象的な仮想モデル**です。国家（Country A〜E）・利得行列・状態遷移確率・係数はすべて架空で、現実の国際関係・紛争や特定国家の行動を予測するものではありません。扱うのは協調・対話・信頼形成・緊張緩和だけです。

## 準備：実験環境（最初に必ず実行）

G2 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（G2 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | G2 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（G2 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                         # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "g2-conflict-cooperation-simulation"    # リポジトリの中の G2 フォルダ（ルートが G2 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_g2_root(p):
    return (Path(p) / "game_theory.py").exists() and (Path(p) / "network_model.py").exists()


def find_g2_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または G2 のルート
        if is_g2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_g2_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("G2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/g2_upload")
    for hit in glob.glob("/content/g2_upload/**/game_theory.py", recursive=True):
        if is_g2_root(Path(hit).parent):
            return Path(hit).parent
    return None


G2_ROOT = find_g2_root()
if G2_ROOT is None:
    raise RuntimeError("G2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(G2_ROOT)
if str(G2_ROOT) not in sys.path:
    sys.path.insert(0, str(G2_ROOT))
if not (G2_ROOT / "textbook" / "fixed_stack.py").exists():
    print("⚠️ textbook/ がありません。network_model.py などの import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import agents as ag
import game_theory as gt
import trust_model as tm
import state_transitions as st
import simulation as sim
import network_model as nm
import cooperation_policies as cp
import evaluation as ev
import conflict_charts as cc

cc.setup_style()                        # 文字を大きくするなら cc.setup_style(1.3)
COUNTRIES = ag.load_countries()         # 架空の国家 A〜E
PARAMS = sim.load_parameters()          # シミュレーションのパラメータ（すべて架空）
M = gt.load_payoff_matrix()             # 利得行列（架空）
IDS = [c["id"] for c in COUNTRIES]
RUNS = PARAMS["monte_carlo_runs"]       # モンテカルロの反復回数
YEARS = PARAMS["simulation_years"]
LABELS = {s: gt.label(s) for s in ag.STRATEGIES}

SAVE_FIGURES = False                    # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", cc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame([{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]))
    except ImportError:
        print(ev.format_table(rows, columns))


SUMMARY_COLUMNS = [("name", "condition", "{}"), ("cooperation_rate", "cooperation", "{:.3f}"),
                   ("cooperation_rate_sd", "sd", "{:.3f}"), ("mutual_cooperation_rate", "mutual coop.", "{:.3f}"),
                   ("cooperation_state_rate", "Cooperation state", "{:.3f}"), ("tension_rate", "Tension", "{:.3f}"),
                   ("conflict_rate", "Conflict", "{:.3f}"), ("conflict_rate_sd", "sd", "{:.3f}")]

print("G2_ROOT:", G2_ROOT)
print("架空の国家:", ", ".join(f"{c['id']}({LABELS[c['strategy']]})" for c in COUNTRIES))
print(f"モンテカルロ {RUNS} 回 × {YEARS} 年、乱数シード {PARAMS['random_seed']}")

## 1. 全探索と Greedy 法（5 国・リング、予算 2 本、対話支援）

目的関数 stability = 1 − 緊張状態率 − 紛争状態率（大きいほど良い）。

In [ ]:
EDGES = nm.make_network("ring", IDS)
BUDGET = PARAMS["policy"]["budget"]            # TODO：支援予算を変更する
OBJECTIVE = "stability"                         # TODO：目的関数を変更する（cooperation / low_conflict）
res = cp.compare(COUNTRIES, EDGES, PARAMS, M, BUDGET, objective=OBJECTIVE)
print(f"支援なし: {res['baseline']:.4f}")
for key in ["exhaustive", "greedy"]:
    x = res[key]
    print(f"{x['algorithm']:18}: {x['objective']:.4f}  gap {x['gap']:.4f}  評価 {x['evaluations']} 回  "
          f"{x['runtime_s']:.2f} s  支援するペア {x['selected']}")
top = sorted(res["exhaustive"]["all"], key=lambda cv: -cv[1])[:5]
print("全探索の上位 5 組:", [(c, round(v, 4)) for c, v in top])

In [ ]:
#@title 目的関数値・実行時間・評価候補数
algs = {"Exhaustive": res["exhaustive"], "Greedy": res["greedy"]}
colors = [cc.ALGO_COLORS["exhaustive"], cc.ALGO_COLORS["greedy"]]
show(cc.plot_bar(["No support"] + list(algs), [res["baseline"]] + [x["objective"] for x in algs.values()],
                 f"Objective ({OBJECTIVE}) by algorithm (model)", OBJECTIVE, ["#b8b7b3"] + colors, "{:.4f}"),
     "05_objective")
show(cc.plot_bar(list(algs), [x["runtime_s"] for x in algs.values()], "Runtime by algorithm (measured on this machine)",
                 "runtime (s)", colors, "{:.2f}"), "05_runtime")
show(cc.plot_bar(list(algs), [x["evaluations"] for x in algs.values()], "Number of evaluated candidates",
                 "evaluations (Monte Carlo runs of the model)", colors, "{:d}"), "05_evaluations")

## 2. 支援対象のエッジと、支援前後の比較

In [ ]:
#@title 支援対象と支援前後の協調率
show(cc.plot_networks([("Exhaustive search", IDS, EDGES), ("Greedy", IDS, EDGES)], "Supported pairs (thick black outline)",
                      highlight=[res["exhaustive"]["selected"], res["greedy"]["selected"]]), "05_supported")
ev_ = res["evaluator"]
before = ev_.summary(())
after = ev_.summary(res["exhaustive"]["selected"])
keys = ["cooperation_rate", "cooperation_state_rate", "tension_rate", "conflict_rate"]
show(cc.plot_grouped(keys, ["before (no support)", "after (exhaustive choice)"],
                     {"before (no support)": [before[k] for k in keys], "after (exhaustive choice)": [after[k] for k in keys]},
                     "Before and after cooperation support (model)", "share",
                     colors={"before (no support)": "#b8b7b3", "after (exhaustive choice)": "#2a78d6"}, ylim=(0, 1.0)),
     "05_before_after")
print("改善量:", {k: round(v, 4) for k, v in ev.policy_improvement(before, after, keys).items()})

## 3. ネットワークの大きさへの依存（候補数と実行時間）

候補数 $C(E, K)$ は、エッジ数 E と予算 K とともに急に増えます。Greedy 法の評価回数は $O(KE)$ です。

In [ ]:
import math
for E_, K_ in [(5, 2), (10, 2), (10, 3), (20, 3), (30, 4), (45, 5)]:
    greedy_evals = sum(E_ - i for i in range(K_))
    print(f"E = {E_:2}, K = {K_}: 全探索 {math.comb(E_, K_):9,} 通り / Greedy 法 {greedy_evals:3} 回")

In [ ]:
#@title 8 国のネットワークでの比較
# 8 国のランダムネットワークで実際に比べる（隣からの波及を強くした例。時間がかかるので反復を減らしている）
import random
rng = random.Random(2)
ids8 = nm.node_names(8)
cs8 = [ag.make_country(i, round(rng.uniform(0.3, 0.8), 2), round(rng.uniform(0.2, 0.8), 2),
                       round(rng.uniform(0.2, 0.8), 2), rng.choice(["conditional", "tit_for_tat"])) for i in ids8]
p8 = sim.with_changes(PARAMS, **{"transition.spillover_effect": 0.5})
e8 = nm.make_network("random", ids8, 0.4, 2)
r8 = cp.compare(cs8, e8, p8, M, 3, runs=15, years=20)
for key in ["exhaustive", "greedy"]:
    x = r8[key]
    print(f"{x['algorithm']:18}: {x['objective']:.4f}  gap {x['gap']:.4f}  評価 {x['evaluations']:3} 回  "
          f"{x['runtime_s']:.2f} s  {x['selected']}")
show(cc.plot_networks([("Exhaustive", ids8, e8), ("Greedy", ids8, e8)], "8 countries, budget 3",
                      highlight=[r8["exhaustive"]["selected"], r8["greedy"]["selected"]]), "05_8_countries")

## 4. 学生向け TODO

In [ ]:
# TODO 1：支援予算を変更する / TODO 2：目的関数を変更する / TODO 3：ネットワークサイズを変更する
# 政策の種類も変えられる："dialogue"・"agreement"・"trust_building"・"economic_cooperation"
for k, objective, policy in [(1, "stability", "dialogue"), (2, "low_conflict", "dialogue"), (2, "stability", "trust_building")]:
    r = cp.compare(COUNTRIES, EDGES, PARAMS, M, k, objective=objective, policy=policy, runs=20, years=30)
    print(f"K={k} {objective:12} {policy:15}: 全探索 {r['exhaustive']['selected']} ({r['exhaustive']['objective']:.4f})"
          f" / Greedy {r['greedy']['selected']} (gap {r['greedy']['gap']:.4f})")

## 考察（チームで記入）

- **TODO**：全探索と Greedy 法の結果が同じだった（違った）のはなぜか。各ペアの効果が独立（足し算）かどうかと関係はあるか
- **TODO**：ネットワークが大きくなると、全探索はいつ現実的でなくなるか（計算量）
- **TODO**：目的関数を変えると、支援すべきペアは変わったか（RQ6）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第4章 スタック・キュー、第5章 再帰、第6章 単純挿入ソート）
- R. Axelrod, *The Evolution of Cooperation*, Basic Books, 1984（繰り返し囚人のジレンマと Tit-for-Tat）
- G2 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_countries.json`、`data/payoff_matrix.json`、`data/simulation_parameters.json`（この教材のための**架空のデータ**）